# 03 — Depuración, población objetivo, variables objetivo y EDA

**Objetivo:** pasar de 113.937 créditos a la población que nos interesa (créditos en dificultad), construir las variables objetivo **A** (¿se recupera algo?) y **B** (¿cuánto?), y entender qué se relaciona con la recuperación.

**Momento de predicción:** cuando el crédito entra a cobranza.

**Resultados que guarda:**
- `data/interim/poblacion_objetivo.parquet`
- `docs/resumen_objetivos.csv`, `docs/chequeos_consistencia.csv`
- Gráficas `docs/figuras/03_*.png`

## 1. Preparación

In [ ]:
import sys
from pathlib import Path

RAIZ = Path.cwd() if (Path.cwd() / 'src').exists() else Path.cwd().parent
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.config import RUTAS
from src.objetivo import (diagnosticar_duplicados, depurar_duplicados, seleccionar_poblacion,
                          chequeos_consistencia, construir_objetivos, resumen_objetivos,
                          marcar_madurez, curva_maduracion, sensibilidad_ventana,
                          VENTANA_MADURACION_MESES)

pd.set_option('display.max_columns', 60)
plt.rcParams['figure.dpi'] = 110

def guardar_figura(fig, nombre):
    fig.savefig(RUTAS['figuras'] / nombre, dpi=150, bbox_inches='tight')

df = pd.read_parquet(RUTAS['data_interim'] / 'prosper_crudo.parquet')
print(f'Cargado: {df.shape[0]:,} filas x {df.shape[1]} columnas')

## 2. ¿Por qué hay créditos repetidos?
Antes de borrar, entendemos qué columnas cambian entre las filas repetidas de un mismo `LoanKey`.

📋 **Pega esta salida.**

In [ ]:
diag = diagnosticar_duplicados(df)
print('Grupos de LoanKey repetidos:', df.loc[df['LoanKey'].duplicated(keep=False), 'LoanKey'].nunique())
print('Filas por grupo (máximo):', df['LoanKey'].value_counts().max())
print('\nColumnas que cambian dentro de los grupos repetidos (n.º de grupos):')
diag

## 3. Depurar: una fila por crédito
**Hallazgo de la sección 2:** las filas repetidas solo difieren en `ProsperScore` (827 grupos). Regla **conservadora**: conservar el `ProsperScore` más bajo (mayor riesgo), porque no hay forma de saber cuál es el correcto.

In [ ]:
df = depurar_duplicados(df)
assert df['LoanKey'].is_unique, 'Todavía hay LoanKey repetidos'
print('✅ LoanKey único')

## 4. Población objetivo: Chargedoff + Defaulted

In [ ]:
pob = seleccionar_poblacion(df)
pob['LoanStatus'].value_counts().to_frame('creditos')

## 5. Chequeos de consistencia de las columnas LP_*
Antes de construir el objetivo verificamos que las columnas se comporten como esperamos. Cada cifra es el **% de créditos** que cumple la condición.

📋 **Pega esta tabla**: de ella depende la definición final del objetivo.

In [ ]:
chequeos = chequeos_consistencia(pob)
chequeos

## 6. Construir las variables objetivo
- **Exposición** = `LP_GrossPrincipalLoss` (capital pendiente al castigo)
- **Recuperación total** = (`LP_GrossPrincipalLoss` − `LP_NetPrincipalLoss`) + `LP_NonPrincipalRecoverypayments`
- **A** = 1 si la recuperación total ≥ 1 USD
- **B** = tasa de recuperación (recuperación / exposición) y monto en USD

📋 **Pega esta tabla.**

**Ajustes tras los chequeos:** recuperación de capital negativa → 0; tasa B acotada a [0, 1]; créditos sin exposición → excluidos del modelado.

In [ ]:
pob = construir_objetivos(pob)
resumen = resumen_objetivos(pob)
resumen.T

## 7. Distribución de los objetivos

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

conteo_A = pob['objetivo_A'].value_counts().sort_index()
axes[0].bar(['No recupera (0)', 'Recupera algo (1)'], conteo_A.values, color=['#C44E52', '#55A868'])
for i, v in enumerate(conteo_A.values):
    axes[0].text(i, v, f'{v:,}\n({v / conteo_A.sum():.1%})', ha='center', va='bottom')
axes[0].set_title('Objetivo A: ¿se recupera algo?')
axes[0].set_ylabel('Créditos')

tasa = pob.loc[pob['objetivo_A'] == 1, 'objetivo_B_tasa'].clip(upper=1)
axes[1].hist(tasa.dropna(), bins=40, color='#4C72B0')
axes[1].set_title('Objetivo B: tasa de recuperación (solo A=1, tope 100%)')
axes[1].set_xlabel('Tasa de recuperación')
axes[1].set_ylabel('Créditos')

plt.tight_layout(); guardar_figura(fig, '03_objetivos.png'); plt.show()

print('Percentiles de la tasa B (solo A=1):')
print(tasa.describe(percentiles=[.1, .25, .5, .75, .9]).round(3))

## 8. ¿Hay censura en los créditos recientes?
Los créditos castigados poco antes de la extracción (2014) **no han tenido tiempo** de mostrar recuperaciones. Si es así, su tasa de recuperación sale artificialmente baja, y eso afecta directamente la validación temporal.

📋 **Pega esta tabla.**

In [ ]:
pob['anio_cierre'] = pob['ClosedDate'].dt.year
pob['anio_originacion'] = pob['LoanOriginationDate'].dt.year

por_cierre = pob.groupby('anio_cierre').agg(creditos=('objetivo_A', 'size'),
                                            pct_recupera=('objetivo_A', 'mean'),
                                            exposicion=('exposicion_castigo', 'sum'),
                                            recuperado=('recuperacion_total', 'sum'))
por_cierre['pct_recupera'] = (por_cierre['pct_recupera'] * 100).round(1)
por_cierre['tasa_cartera_pct'] = (por_cierre['recuperado'] / por_cierre['exposicion'] * 100).round(1)

por_orig = pob.groupby('anio_originacion').agg(creditos=('objetivo_A', 'size'),
                                              pct_recupera=('objetivo_A', 'mean'))
por_orig['pct_recupera'] = (por_orig['pct_recupera'] * 100).round(1)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
por_cierre['pct_recupera'].plot(marker='o', ax=axes[0], title='% que recupera algo, por año de cierre (castigo)')
por_orig['pct_recupera'].plot(marker='o', ax=axes[1], color='#DD8452', title='% que recupera algo, por año de originación')
for ax in axes: ax.set_ylabel('%'); ax.grid(alpha=.3)
plt.tight_layout(); guardar_figura(fig, '03_recuperacion_por_anio.png'); plt.show()

print('Por año de cierre:'); display(por_cierre)
print('Por año de originación:'); display(por_orig)

## 8b. Ventana de maduración (corrección de la censura)
La sección 8 mostró que los créditos cerrados en 2013–2014 recuperan mucho menos: **no han tenido tiempo**. Medimos cómo crece el % que recupera según los meses de observación después del cierre. Donde la curva se estabiliza está la ventana mínima.

- **Fecha de corte** = fecha más reciente del dataset (proxy de la fecha de extracción).
- Créditos **maduros** (≥ ventana) → entrenamiento y validación.
- Créditos **inmaduros** → no se usan para entrenar; quedan como cartera para demostrar la priorización en la app.

📋 **Pega las dos tablas.**

In [ ]:
fecha_corte = df[['ListingCreationDate', 'LoanOriginationDate', 'ClosedDate']].max().max()
print('Fecha de corte (proxy de extracción):', fecha_corte.date())

pob = marcar_madurez(pob, fecha_corte)
curva = curva_maduracion(pob)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(range(len(curva)), curva['pct_recupera'], marker='o')
ax.set_xticks(range(len(curva)), curva.index, rotation=45, ha='right')
ax.axvline(VENTANA_MADURACION_MESES / 6 - 0.5, color='red', ls='--', label=f'ventana = {VENTANA_MADURACION_MESES} meses')
ax.set_xlabel('Meses de observación después del cierre'); ax.set_ylabel('% que recupera algo')
ax.set_title('Curva de maduración de la recuperación'); ax.legend(); ax.grid(alpha=.3)
plt.tight_layout(); guardar_figura(fig, '03_curva_maduracion.png'); plt.show()

print('Curva de maduración:'); display(curva)
print('\nSensibilidad a la ventana elegida:'); display(sensibilidad_ventana(pob))

## 9. Variable a revisar: `LoanFirstDefaultedCycleNumber`
¿Está disponible para nuestra población? Si sí, puede ser una predictora muy valiosa (cuántas cuotas pagó antes de caer).

In [ ]:
col = 'LoanFirstDefaultedCycleNumber'
print(pob.groupby('LoanStatus')[col].apply(lambda s: f'{s.notna().mean():.1%} con dato'))
print(pob[col].describe().round(1))

## 10. EDA: ¿qué se relaciona con la recuperación?
Usamos **solo créditos maduros** (para que la censura no distorsione) y comparamos el **% que recupera algo (A)** y la **tasa de recuperación de la cartera** por grupos. Es exploración, no causalidad: el modelo confirmará qué variables pesan.

Para la calificación de riesgo unimos las dos épocas: `CreditGrade` (antigua) y `ProsperRating (Alpha)` (reciente).

In [ ]:
pob['calificacion'] = pob['ProsperRating (Alpha)'].fillna(pob['CreditGrade'])
eda = pob[pob['en_modelado']].copy()
print(f'EDA sobre {len(eda):,} créditos maduros con exposición válida')
eda['monto_quintil'] = pd.qcut(eda['LoanOriginalAmount'], 5,
                             labels=['Q1 (menor)', 'Q2', 'Q3', 'Q4', 'Q5 (mayor)'])

def tabla_recuperacion(var):
    t = eda.groupby(var, observed=True).agg(creditos=('objetivo_A', 'size'),
                                           pct_recupera=('objetivo_A', 'mean'),
                                           exposicion=('exposicion_castigo', 'sum'),
                                           recuperado=('recuperacion_total', 'sum'))
    t['pct_recupera'] = (t['pct_recupera'] * 100).round(1)
    t['tasa_cartera_pct'] = (t['recuperado'] / t['exposicion'].where(t['exposicion'] > 0) * 100).round(1)
    return t[t['creditos'] >= 30].drop(columns=['exposicion', 'recuperado'])

variables = ['calificacion', 'Term', 'IncomeRange', 'EmploymentStatus',
             'IsBorrowerHomeowner', 'IncomeVerifiable', 'monto_quintil']
# Plazo agrupado con el año de cierre: los créditos a 60 meses existen solo desde 2010
print(pd.crosstab(eda['anio_cierre'], eda['Term']))
tablas = {v: tabla_recuperacion(v) for v in variables}

fig, axes = plt.subplots(len(variables), 1, figsize=(9, 3.2 * len(variables)))
for ax, v in zip(axes, variables):
    t = tablas[v]
    ax.barh(t.index.astype(str), t['pct_recupera'], color='#4C72B0')
    ax.axvline(eda['objetivo_A'].mean() * 100, color='gray', ls='--', lw=1, label='promedio')
    ax.set_title(f'% que recupera algo por {v}'); ax.set_xlabel('%')
plt.tight_layout(); guardar_figura(fig, '03_eda_recuperacion.png'); plt.show()

📋 **Pega estas tablas** (son las que alimentan el análisis del documento y el resultado de la Fase 3).

In [ ]:
for v, t in tablas.items():
    print(f'\n===== {v} =====')
    print(t.to_string())

## 11. Guardar resultados

In [ ]:
print(pob.groupby(['madura', 'exposicion_valida']).size().rename('creditos').to_frame())
print(f"\n➡️ Créditos para modelar (en_modelado): {pob['en_modelado'].sum():,}")

salida = RUTAS['data_interim'] / 'poblacion_objetivo.parquet'
pob.to_parquet(salida, index=False)
resumen.to_csv(RUTAS['docs'] / 'resumen_objetivos.csv', encoding='utf-8-sig')
chequeos.to_csv(RUTAS['docs'] / 'chequeos_consistencia.csv', encoding='utf-8-sig')

print('✅ Guardado:', salida.relative_to(RUTAS['raiz']), f'({len(pob):,} filas)')
print('   docs/resumen_objetivos.csv, docs/chequeos_consistencia.csv, docs/figuras/03_*.png')